In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## Hakbang 1: Tukuyin ang mga Pydantic na Modelo para sa Mga Istrakturang Output

Itong mga modelo ang nagtatakda ng **schema** na ibabalik ng mga ahente. Ang paggamit ng `response_format` kasama ang Pydantic ay nagsisiguro ng:
- ✅ Ligtas sa uri ng datos na pagkuha
- ✅ Awtomatikong beripikasyon
- ✅ Walang error sa pag-parse mula sa mga malayang tugon
- ✅ Madaling kundisyunal na pag-ruta batay sa mga patlang


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## Hakbang 2: Lumikha ng Kasangkapan sa Pag-book ng Hotel

Ang kasangkapang ito ang tatawagin ng **availability_agent** upang suriin kung may mga bakanteng kuwarto. Ginagamit namin ang `@ai_function` decorator upang:
- I-convert ang isang Python function bilang isang AI-callable na kasangkapan
- Awtomatikong gumawa ng JSON schema para sa LLM
- Pamahalaan ang beripikasyon ng mga parameter
- Paganahin ang awtomatikong pagtawag ng mga ahente

Para sa demo na ito:
- **Stockholm, Seattle, Tokyo, London, Amsterdam** → May mga kuwarto ✅
- **Lahat ng iba pang mga lungsod** → Walang mga kuwarto ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## Hakbang 3: Tukuyin ang Mga Function ng Kondisyon para sa Routing

Sinusuri ng mga function na ito ang tugon ng ahente at tinutukoy kung aling landas ang tatahakin sa workflow.

**Pangunahing Pattern:**
1. Suriin kung ang mensahe ay `AgentExecutorResponse`
2. I-parse ang structured output (Pydantic model)
3. Ibalik ang `True` o `False` upang kontrolin ang routing

Susuriin ng workflow ang mga kondisyong ito sa **edges** upang magpasya kung aling executor ang tatawagin sunod.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## Hakbang 4: Lumikha ng Custom Display Executor

Ang mga executor ay mga bahagi ng workflow na nagsasagawa ng mga transformasyon o mga side effect. Ginagamit natin ang `@executor` decorator upang lumikha ng isang custom executor na nagpapakita ng panghuling resulta.

**Mga Pangunahing Konsepto:**
- `@executor(id="...")` - Nagpaparehistro ng isang function bilang executor ng workflow
- `WorkflowContext[Never, str]` - Mga type hint para sa input/output
- `ctx.yield_output(...)` - Nagbibigay ng panghuling resulta ng workflow


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## Hakbang 5: I-load ang Mga Variable ng Kapaligiran

I-configure ang LLM client. Ang halimbawang ito ay gumagana sa:
- **Microsoft Foundry** / **Azure OpenAI** (Responses API — inirerekomenda)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## Hakbang 6: Lumikha ng mga AI Agent na may Mga Istrakturang Output

Gumagawa kami ng **tatlong espesyal na ahente**, bawat isa ay nakabalot sa isang `AgentExecutor`:

1. **availability_agent** - Sinusuri ang availability ng hotel gamit ang tool
2. **alternative_agent** - Nagmumungkahi ng mga alternatibong lungsod (kapag walang kuwarto)
3. **booking_agent** - Nag-uudyok ng pag-book (kapag may mga kuwarto)

**Mga Pangunahing Katangian:**
- `tools=[hotel_booking]` - Nagbibigay ng tool sa agent
- `response_format=PydanticModel` - Pinipilit ang istrakturang output na JSON
- `AgentExecutor(..., id="...")` - Nakabalot ang agent para sa paggamit sa workflow


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## Hakbang 7: Ibuo ang Workflow gamit ang Kondisyonal na Mga Daan

Ngayon gagamitin natin ang `WorkflowBuilder` upang buuin ang grap na may kondisiynal na pag-ruta:

**Istruktura ng Workflow:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**Pangunahing Paraan:**
- `.set_start_executor(...)` - Nagtatakda ng panimulang punto
- `.add_edge(from, to, condition=...)` - Nagdaragdag ng kondisyonal na daan
- `.build()` - Pinagtatapos ang workflow


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## Hakbang 8: Patakbuhin ang Test Case 1 - Lungsod NA WALANG Availability (Paris)

Subukan natin ang path na **walang availability** sa pamamagitan ng paghingi ng mga hotel sa Paris (na walang mga kuwarto sa aming simulasyon).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## Hakbang 9: Patakbuhin ang Test Case 2 - Lungsod NA MAY Availability (Stockholm)

Ngayon subukan natin ang **availability** na landas sa pamamagitan ng paghingi ng mga hotel sa Stockholm (na may mga kuwarto sa ating simulation).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## Mga Pangunahing Natutunan at Mga Susunod na Hakbang

### ✅ Mga Natutunan Mo:

1. **WorkflowBuilder Pattern**
   - Gamitin ang `.set_start_executor()` para tukuyin ang entry point
   - Gamitin ang `.add_edge(from, to, condition=...)` para sa kundisyunal na pagtutok
   - Tawagin ang `.build()` para tapusin ang workflow

2. **Kundisyunal na Pagtutok**
   - Tinutukoy ng mga condition functions ang `AgentExecutorResponse`
   - I-parse ang mga structured outputs para gumawa ng mga desisyon sa pagtutok
   - Ibalik ang `True` para i-activate ang edge, `False` para laktawan ito

3. **Integrasyon ng Tool**
   - Gamitin ang `@ai_function` para gawing AI tools ang Python functions
   - Awtomatikong ginagamit ng mga ahente ang mga tools kapag kinakailangan
   - Nagbabalik ang mga tools ng JSON na puwedeng i-parse ng mga ahente

4. **Mga Structured Outputs**
   - Gamitin ang mga Pydantic models para sa type-safe na pagkuha ng data
   - Itakda ang `response_format=MyModel` kapag gumagawa ng mga ahente
   - I-parse ang mga tugon gamit ang `Model.model_validate_json()`

5. **Custom Executors**
   - Gamitin ang `@executor(id="...")` para gumawa ng mga bahagi ng workflow
   - Maaaring baguhin ng mga executor ang data o magsagawa ng side effects
   - Gamitin ang `ctx.yield_output()` para maglabas ng mga resulta ng workflow

### 🚀 Mga Aplikasyon sa Totoong Mundo:

- **Pag-book ng Paglalakbay**: Suriin ang availability, magmungkahi ng alternatibo, ikumpara ang mga opsyon
- **Serbisyo sa Customer**: I-route batay sa uri ng isyu, damdamin, prayoridad
- **E-commerce**: Suriin ang imbentaryo, magmungkahi ng alternatibo, proseso ng mga order
- **Pagsusuri ng Nilalaman**: I-route batay sa toxicity scores, mga flag ng user
- **Mga Approval Workflow**: I-route batay sa halaga, papel ng user, antas ng panganib
- **Multi-stage na Pagpoproseso**: I-route batay sa kalidad ng data, kumpletong impormasyon

### 📚 Mga Susunod na Hakbang:

- Magdagdag ng mas kumplikadong mga kundisyon (maramihang kriteriya)
- Ipatupad ang mga loop gamit ang pamamahala ng estado ng workflow
- Magdagdag ng mga sub-workflow para sa mga reusable na bahagi
- Isama sa mga totoong API (pag-book ng hotel, mga sistema ng imbentaryo)
- Magdagdag ng paghawak sa error at mga fallback na landas
- I-visualize ang mga workflow gamit ang mga built-in na visualization tools


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
